In [3]:
import zipfile
import pandas as pd

In [4]:
# Location of our Kaggle dataset
zip_path = "../data/SWPK.zip"

# Open the ZIP and list the CSV files inside
with zipfile.ZipFile(zip_path, "r") as zip_file:
    csv_files = [
        filename
        for filename in zip_file.namelist()
        if filename.endswith(".csv")
    ]

print("CSV files found:", len(csv_files))
print(csv_files)

CSV files found: 22
['datadump_s5-000.csv', 'datadump_s5-001.csv', 'datadump_s5-002.csv', 'datadump_s5-003.csv', 'datadump_s5-004.csv', 'datadump_s5-005.csv', 'datadump_s5-006.csv', 'datadump_s5-007.csv', 'datadump_s5-008.csv', 'datadump_s5-009.csv', 'datadump_s5-010.csv', 'datadump_s5-011.csv', 'datadump_s5-012.csv', 'datadump_s5-013.csv', 'datadump_s5-014.csv', 'datadump_s5-015.csv', 'datadump_s5-016.csv', 'datadump_s5-017.csv', 'datadump_s5-018.csv', 'datadump_s5-019.csv', 'datadump_s5-020.csv', 'datadump_s5-021.csv']


In [5]:
# Columns needed for our machine-learning project
needed_columns = [
    "matchid",
    "roundnumber",
    "mapname",
    "objectivelocation",
    "winrole",
    "role",
    "operator"
]

print("Columns we will use:")
print(needed_columns)

Columns we will use:
['matchid', 'roundnumber', 'mapname', 'objectivelocation', 'winrole', 'role', 'operator']


In [6]:
# Store the processed rounds from each CSV file
all_rounds = []

# Process each CSV file one at a time
with zipfile.ZipFile(zip_path, "r") as zip_file:

    for i, filename in enumerate(csv_files, start=1):

        print(f"Processing {i}/{len(csv_files)}: {filename}")

        # Open one CSV directly from the ZIP
        with zip_file.open(filename) as csv_file:

            df_part = pd.read_csv(
                csv_file,
                usecols=needed_columns
            )

        # Store the rounds found in this file
        file_rounds = []

        # Group players by match and round WITHIN this file
        for (matchid, roundnumber), group in df_part.groupby(
            ["matchid", "roundnumber"]
        ):

            # Get the five attackers
            attackers = group[
                group["role"] == "Attacker"
            ]["operator"].tolist()

            # Get the five defenders
            defenders = group[
                group["role"] == "Defender"
            ]["operator"].tolist()

            # Only keep complete 5v5 rounds
            if len(attackers) != 5 or len(defenders) != 5:
                continue

            # These values are the same for every player in the round
            mapname = group["mapname"].iloc[0]
            objectivelocation = group["objectivelocation"].iloc[0]
            winrole = group["winrole"].iloc[0]

            # 1 = attackers won
            # 0 = defenders won
            attack_win = 1 if winrole == "Attacker" else 0

            file_rounds.append({
                "source_file": filename,
                "matchid": matchid,
                "roundnumber": roundnumber,
                "mapname": mapname,
                "objectivelocation": objectivelocation,
                "attackers": attackers,
                "defenders": defenders,
                "attack_win": attack_win
            })

        # Convert this file's results into a DataFrame
        file_rounds = pd.DataFrame(file_rounds)

        print("Complete rounds:", len(file_rounds))

        all_rounds.append(file_rounds)


# Combine all 21 CSV files
round_df_full = pd.concat(
    all_rounds,
    ignore_index=True
)


# Clean operator names
def clean_operator_name(operator):
    operator_name = operator.split("-")[-1]

    # Historical dataset name for Recruit
    if operator_name == "RESERVE":
        return "RECRUIT"

    return operator_name


round_df_full["attackers"] = round_df_full["attackers"].apply(
    lambda operators: [
        clean_operator_name(operator)
        for operator in operators
    ]
)

round_df_full["defenders"] = round_df_full["defenders"].apply(
    lambda operators: [
        clean_operator_name(operator)
        for operator in operators
    ]
)


print("\n==============================")
print("FULL DATASET COMPLETE")
print("==============================")
print("Total complete rounds:", len(round_df_full))

Processing 1/22: datadump_s5-000.csv
Complete rounds: 237993
Processing 2/22: datadump_s5-001.csv
Complete rounds: 244039
Processing 3/22: datadump_s5-002.csv
Complete rounds: 231777
Processing 4/22: datadump_s5-003.csv
Complete rounds: 234736
Processing 5/22: datadump_s5-004.csv
Complete rounds: 238728
Processing 6/22: datadump_s5-005.csv
Complete rounds: 247232
Processing 7/22: datadump_s5-006.csv
Complete rounds: 236541
Processing 8/22: datadump_s5-007.csv
Complete rounds: 232612
Processing 9/22: datadump_s5-008.csv
Complete rounds: 228447
Processing 10/22: datadump_s5-009.csv
Complete rounds: 239224
Processing 11/22: datadump_s5-010.csv
Complete rounds: 234906
Processing 12/22: datadump_s5-011.csv
Complete rounds: 236870
Processing 13/22: datadump_s5-012.csv
Complete rounds: 231904
Processing 14/22: datadump_s5-013.csv
Complete rounds: 239322
Processing 15/22: datadump_s5-014.csv
Complete rounds: 236074
Processing 16/22: datadump_s5-015.csv
Complete rounds: 232071
Processing 17/22:

In [7]:
# Check whether each source file + match + round is unique

duplicates = round_df_full.duplicated(
    subset=["source_file", "matchid", "roundnumber"]
).sum()

print("Duplicate rounds:", duplicates)

Duplicate rounds: 0


In [14]:
import pyarrow as pa
import pyarrow.parquet as pq

# Convert the DataFrame to an Arrow table
table = pa.Table.from_pandas(
    round_df_full,
    preserve_index=False
)

# Save the table
pq.write_table(
    table,
    "../data/round_df_full.parquet"
)

print("Checkpoint saved successfully!")

Checkpoint saved successfully!


In [15]:
import pyarrow.parquet as pq

table = pq.read_table(
    "../data/round_df_full.parquet"
)

print("Rows in checkpoint:", table.num_rows)

Rows in checkpoint: 5082146


In [16]:
print("Total rounds:", len(round_df_full))
print("Unique matches:", round_df_full["matchid"].nunique())
print("Unique maps:", round_df_full["mapname"].nunique())
print("Unique sites:", round_df_full["objectivelocation"].nunique())

print("\nAttack vs Defense wins:")
print(round_df_full["attack_win"].value_counts())

print("\nWin percentages:")
print(
    round_df_full["attack_win"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Total rounds: 5082146
Unique matches: 1181156
Unique maps: 16
Unique sites: 142

Attack vs Defense wins:
attack_win
1    2543618
0    2538528
Name: count, dtype: int64

Win percentages:
attack_win
1    50.05
0    49.95
Name: proportion, dtype: float64


In [17]:
# Get every unique operator used by attackers
attack_operators = sorted(
    set(
        operator
        for operators in round_df_full["attackers"]
        for operator in operators
    )
)

# Get every unique operator used by defenders
defense_operators = sorted(
    set(
        operator
        for operators in round_df_full["defenders"]
        for operator in operators
    )
)

print("Attack operators:", len(attack_operators))
print(attack_operators)

print("\nDefense operators:", len(defense_operators))
print(defense_operators)

Attack operators: 16
['ASH', 'BLACKBEARD', 'BLITZ', 'BUCK', 'CAPITAO', 'FUZE', 'GLAZ', 'HIBANA', 'IQ', 'JACKAL', 'MONTAGNE', 'RECRUIT', 'SLEDGE', 'THATCHER', 'THERMITE', 'TWITCH']

Defense operators: 16
['BANDIT', 'CASTLE', 'CAVEIRA', 'DOC', 'ECHO', 'FROST', 'JAGER', 'KAPKAN', 'MIRA', 'MUTE', 'PULSE', 'RECRUIT', 'ROOK', 'SMOKE', 'TACHANKA', 'VALKYRIE']


In [18]:
from collections import Counter

attack_counts = Counter(
    operator
    for operators in round_df_full["attackers"]
    for operator in operators
)

defense_counts = Counter(
    operator
    for operators in round_df_full["defenders"]
    for operator in operators
)

print("Most common attackers:")
print(attack_counts.most_common(15))

print("\nMost common defenders:")
print(defense_counts.most_common(15))

Most common attackers:
[('ASH', 3232341), ('TWITCH', 2561153), ('HIBANA', 2557492), ('THERMITE', 2446272), ('FUZE', 2040477), ('JACKAL', 1983908), ('SLEDGE', 1946514), ('GLAZ', 1738656), ('THATCHER', 1622384), ('BUCK', 1216447), ('BLACKBEARD', 935225), ('MONTAGNE', 892365), ('CAPITAO', 781121), ('IQ', 767688), ('BLITZ', 364939)]

Most common defenders:
[('JAGER', 3494748), ('BANDIT', 2682877), ('CAVEIRA', 2191108), ('VALKYRIE', 2153075), ('MUTE', 1878548), ('SMOKE', 1732514), ('FROST', 1707721), ('ROOK', 1702372), ('PULSE', 1689234), ('MIRA', 1669322), ('DOC', 1330461), ('KAPKAN', 1019121), ('CASTLE', 1001782), ('ECHO', 577806), ('RECRUIT', 394830)]


In [19]:
print("All attacker operator counts:")
for operator, count in sorted(attack_counts.items()):
    print(f"{operator:12} {count:,}")

print("\nAll defender operator counts:")
for operator, count in sorted(defense_counts.items()):
    print(f"{operator:12} {count:,}")

All attacker operator counts:
ASH          3,232,341
BLACKBEARD   935,225
BLITZ        364,939
BUCK         1,216,447
CAPITAO      781,121
FUZE         2,040,477
GLAZ         1,738,656
HIBANA       2,557,492
IQ           767,688
JACKAL       1,983,908
MONTAGNE     892,365
RECRUIT      323,748
SLEDGE       1,946,514
THATCHER     1,622,384
THERMITE     2,446,272
TWITCH       2,561,153

All defender operator counts:
BANDIT       2,682,877
CASTLE       1,001,782
CAVEIRA      2,191,108
DOC          1,330,461
ECHO         577,806
FROST        1,707,721
JAGER        3,494,748
KAPKAN       1,019,121
MIRA         1,669,322
MUTE         1,878,548
PULSE        1,689,234
RECRUIT      394,830
ROOK         1,702,372
SMOKE        1,732,514
TACHANKA     185,211
VALKYRIE     2,153,075


In [20]:
# See how many unique map + site combinations exist

map_site = (
    round_df_full["mapname"]
    + "__"
    + round_df_full["objectivelocation"]
)

print("Unique map + site combinations:", map_site.nunique())

Unique map + site combinations: 160


In [21]:
# Show some examples

print(
    map_site.drop_duplicates()
    .sort_values()
    .head(30)
)

4                                BANK__ARCHIVES
74                             BANK__CEO_OFFICE
319           BANK__EXECUTIVE_LOUNGE-CEO_OFFICE
0                                 BANK__LOCKERS
320                     BANK__LOCKERS-CCTV_ROOM
5                               BANK__OPEN_AREA
2318                           BANK__STAFF_ROOM
1014                 BANK__STAFF_ROOM-OPEN_AREA
1323                      BANK__TELLER'S_OFFICE
322              BANK__TELLERS'_OFFICE-ARCHIVES
1320                                BANK__VAULT
34                       BARTLETT_U.__CLASSROOM
295              BARTLETT_U.__CLASSROOM-LIBRARY
490                        BARTLETT_U.__KITCHEN
69              BARTLETT_U.__KITCHEN-PIANO_ROOM
38                         BARTLETT_U.__LIBRARY
39                          BARTLETT_U.__LOUNGE
3556                   BARTLETT_U.__MAIN_OFFICE
37                      BARTLETT_U.__MODEL_HALL
296           BARTLETT_U.__READING_ROOM-LIBRARY
1269     BARTLETT_U.__ROWING_MUSEUM-TROP

In [22]:
# Make a copy of our full round dataset
features_df = round_df_full.copy()


# --------------------------------
# 1. ATTACKER OPERATOR FEATURES
# --------------------------------

for operator in attack_operators:
    features_df[f"ATTACK_{operator}"] = features_df["attackers"].apply(
        lambda team: int(operator in team)
    )


# --------------------------------
# 2. DEFENDER OPERATOR FEATURES
# --------------------------------

for operator in defense_operators:
    features_df[f"DEFENSE_{operator}"] = features_df["defenders"].apply(
        lambda team: int(operator in team)
    )


# --------------------------------
# 3. MAP FEATURES
# --------------------------------

map_features = pd.get_dummies(
    features_df["mapname"],
    prefix="MAP",
    dtype="int8"
)


# --------------------------------
# 4. MAP + SITE FEATURES
# --------------------------------

features_df["map_site"] = (
    features_df["mapname"]
    + "__"
    + features_df["objectivelocation"]
)

map_site_features = pd.get_dummies(
    features_df["map_site"],
    prefix="MAPSITE",
    dtype="int8"
)


# Add the categorical features
features_df = pd.concat(
    [
        features_df,
        map_features,
        map_site_features
    ],
    axis=1
)


print("Feature dataset shape:", features_df.shape)

Feature dataset shape: (5082146, 217)


In [23]:
print("Rows:", len(features_df))
print("Columns:", len(features_df.columns))

print("\nMemory usage:")
print(
    round(
        features_df.memory_usage(deep=True).sum() / (1024 ** 3),
        2
    ),
    "GB"
)

Rows: 5082146
Columns: 217

Memory usage:
4.61 GB


In [25]:
import pyarrow as pa
import pyarrow.parquet as pq

# Convert the feature DataFrame to an Arrow table
table = pa.Table.from_pandas(
    features_df,
    preserve_index=False
)

# Save the feature dataset
pq.write_table(
    table,
    "../data/features_df.parquet"
)

print("Feature checkpoint saved!")

Feature checkpoint saved!


In [26]:
table = pq.read_table(
    "../data/features_df.parquet"
)

print("Rows:", table.num_rows)
print("Columns:", table.num_columns)

Rows: 5082146
Columns: 217
